# Qwen2.5-Coder C++ Review — QLoRA Training on Kaggle

Run top to bottom. **Cell 2 (`RUN CONTROL`) is the only cell you normally edit.**

**Fresh run** — `FRESH_START = True`. Anything already in `OUTPUT_DIR` is moved to
`outputs/.../archive/run-<timestamp>/` (moved, not deleted) and training begins at step 0.

**Continuing** — `FRESH_START = False`. Checkpoints are imported from `PREVIOUS_RUN_DIR`
and training picks up at the exact step it stopped at.

Either way, `training.resume_mode` stays `auto`: if this Kaggle session is killed
mid-run, re-running just the training cell continues where it left off. Freshness is a
property of the output directory, not a setting — so a crash can never be mistaken for
a request to start over.

In [ ]:
# ============================================================================
# RUN CONTROL — the only cell you normally edit
# ============================================================================
# True  -> train from step 0. Existing output is archived; no checkpoints imported.
# False -> continue a previous run by importing its checkpoints from PREVIOUS_RUN_DIR.
FRESH_START = True

# The task-tagged mixture from scripts/build_task_mixture.py, not the raw
# merged file. Update the dataset slug below to whatever you named the upload.
DATASET_FILE     = '/kaggle/input/datasets/saffiullah892/mydataset01/task_mixture.jsonl'
PROJECT_INPUT    = '/kaggle/input/datasets/saffiullah892/my-projectfiles01'
OUTPUT_DIR       = '/kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora'
PREVIOUS_RUN_DIR = '/kaggle/input/datasets/saffiullah892/qwen2-5-01/outputs/qwen2.5-coder-1.5b-cpp-review-qlora'

import shutil
import time
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

if FRESH_START:
    existing = [path for path in output_dir.iterdir() if path.name != 'archive']
    if existing:
        archive = output_dir / 'archive' / time.strftime('run-%Y%m%d-%H%M%S')
        archive.mkdir(parents=True, exist_ok=True)
        for path in existing:
            shutil.move(str(path), str(archive / path.name))
        print(f'Archived {len(existing)} item(s) from the previous run to {archive}')
    print('FRESH START — training begins at step 0')
    print('After this run starts, set FRESH_START = False so a session restart resumes')
    print('instead of archiving your progress.')
else:
    source = Path(PREVIOUS_RUN_DIR)
    assert source.is_dir(), f'PREVIOUS_RUN_DIR not found: {source}'
    imported = 0
    for path in sorted(source.glob('checkpoint-*')):
        target = output_dir / path.name
        if not target.exists():
            shutil.copytree(path, target)
            imported += 1
    print(f'Imported {imported} checkpoint(s) from {source}')
    print('RESUME — training continues from the newest complete checkpoint')

present = sorted(path.name for path in output_dir.glob('checkpoint-*'))
print('Checkpoints in the output dir:', present or 'none')

In [ ]:
!nvidia-smi
!python --version
!python -m pip install --upgrade uv

## Copy Project to Writable Storage

Kaggle mounts `/kaggle/input` as read-only. Training needs to write configs, checkpoints, adapters, `.pth` files, and ONNX exports, so the project is copied to `/kaggle/working/project-files`.

In [ ]:
import shutil
from pathlib import Path

project_input = Path(PROJECT_INPUT)
work_repo = Path('/kaggle/working/project-files')

assert project_input.is_dir(), f'Project folder not found: {project_input}'
assert Path(DATASET_FILE).is_file(), f'Training dataset not found: {DATASET_FILE}'

# The dataset must hold the project tree itself. Kaggle expands an archive only
# when it is uploaded as one to extract; a .zip added as a plain file arrives as
# a .zip, and then nothing below is the code this notebook needs.
marker = project_input / 'src' / 'qwen_cpp_review' / 'prompt.py'
if not marker.exists():
    print(f'{project_input}\ndoes not contain src/qwen_cpp_review/prompt.py. It contains:\n')
    for item in sorted(project_input.iterdir())[:25]:
        print('   ', item.name + ('/' if item.is_dir() else ''))
    raise SystemExit(
        'Upload the CONTENTS of dist/kaggle-project-files.zip (let Kaggle extract it), '
        'not the zip as a file.'
    )

# /kaggle/working survives between sessions. A previous run's copy sits here and
# shadows the new upload, so replace it rather than merging into it - that is
# what makes a correct re-upload look like it changed nothing.
%cd /kaggle/working
if work_repo.exists():
    shutil.rmtree(work_repo)
    print('Removed the previous working copy')
shutil.copytree(project_input, work_repo)
%cd /kaggle/working/project-files

assert Path('pyproject.toml').exists(), 'pyproject.toml missing after copy'
print('Project copied to:', work_repo)
print('Training dataset:', DATASET_FILE)

## Verify the Project Files Support Resume

Kaggle serves project files from a dataset snapshot, which can lag behind the repository.
This fails fast rather than training for hours with the old code.

In [ ]:
import time
from pathlib import Path

# Every marker below must be present in the project files this notebook copied
# into /kaggle/working. If one is missing the upload is stale, and the failure
# it prevents is silent, so this cell stops the run instead of warning.
REQUIRED = [
    ('src/qwen_cpp_review/resume.py', None, 'the resume rewrite'),
    ('src/qwen_cpp_review/config.py', 'resume_mode', 'the resume config fields'),
    ('src/qwen_cpp_review/trainer.py', 'resolve_resume_plan', 'the resume rewrite'),
    ('src/qwen_cpp_review/prompt.py', 'TASKS = {', 'the task registry'),
    ('src/qwen_cpp_review/prompt.py', 'def has_field', 'the None-valued field guard'),
    ('src/qwen_cpp_review/prompt.py', 'def resolve_output_fields', 'per-task field resolution'),
    ('src/qwen_cpp_review/identifier_augmentation.py', 'apply_mapping_to_row', 'the anchor-safe rename'),
]

stale = []
for relative, marker, what in REQUIRED:
    path = Path(relative)
    if not path.exists():
        stale.append((relative, f'file missing ({what})'))
    elif marker and marker not in path.read_text():
        stale.append((relative, f'missing {what}'))

if stale:
    print('STALE PROJECT FILES\n')
    for relative, why in stale:
        path = Path(relative)
        when = time.strftime('%Y-%m-%d %H:%M', time.localtime(path.stat().st_mtime)) if path.exists() else '-'
        print(f'  {relative:<48} {why}   (mtime {when})')

    print(f'\nWorking copy : {Path.cwd()}')
    print(f'Mounted from : {PROJECT_INPUT}')
    print('\nFix, in order:')
    print('  1. Upload dist/kaggle-project-files.zip as a NEW VERSION of the')
    print('     project-files dataset (not a new dataset, unless you also change')
    print('     PROJECT_INPUT above).')
    print('  2. In this notebook, open the right-hand Input panel and confirm the')
    print('     dataset shows the new version. Kaggle pins the version that was')
    print('     attached when the session started.')
    print('  3. Restart the session (Run -> Restart & clear cell outputs), then run')
    print('     from the top. /kaggle/input is mounted at session start, so a new')
    print('     dataset version is invisible to a session that is already running.')
    raise SystemExit('Stale project files - see the steps above')

print('Resume-capable, task-aware project files detected')
print('Working copy:', Path.cwd())

## Install Dependencies with uv

In [ ]:
!mkdir -p /kaggle/temp/uv-cache /kaggle/temp/project-venv /kaggle/temp/hf-cache /kaggle/temp/hf-datasets
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache UV_LINK_MODE=copy uv sync --extra gpu --extra export --extra dev
!du -sh /kaggle/temp/project-venv /kaggle/temp/uv-cache /kaggle/working/project-files || true

## Configure Training

Writes the Kaggle paths into `configs/train_qlora.yaml`. `resume_mode` is left on `auto`
so an interrupted session resumes exactly; `optim` is pinned because a checkpoint's
optimizer state can only be reloaded by the optimizer that wrote it.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/train_qlora.yaml')
config = yaml.safe_load(config_path.read_text())

config['data']['data_files'] = [DATASET_FILE]
config['data']['cache_dir'] = '/kaggle/temp/hf-datasets'
config['data']['identifier_augmentation'] = True
config['data']['identifier_augmentation_copies'] = 1

training = config['training']
training['output_dir'] = OUTPUT_DIR
# auto = continue from the newest usable checkpoint, start at 0 when there is none.
training['resume_mode'] = 'auto'
training['resume_from_checkpoint'] = None
training['initial_adapter_path'] = None
training['resume_auto_fallback'] = True
# Keep this fixed for the whole run — changing it makes saved optimizer state unloadable.
training['optim'] = 'adamw_torch'
training['logging_steps'] = 5
# Save/eval every 50 optimizer steps: a killed session costs at most 50 steps.
# save_total_limit=3 leaves room for the newest checkpoint plus the best one.
training['save_steps'] = 50
training['eval_steps'] = 50
training['save_total_limit'] = 3
training['packing'] = False
training['gradient_checkpointing_use_reentrant'] = False
training['ddp_find_unused_parameters'] = False

config_path.write_text(yaml.safe_dump(config, sort_keys=False))

print('Dataset      :', config['data']['data_files'])
print('Output dir   :', training['output_dir'])
print('Resume mode  :', training['resume_mode'])
print('Optimizer    :', training['optim'])
print('Epochs       :', training['num_train_epochs'])

## Optional Sanity Check

This checks that the dataset file is readable and shows the first row keys.

In [ ]:
import json
from collections import Counter
from pathlib import Path

dataset_path = Path(DATASET_FILE)
tasks = Counter()
rows = 0
with dataset_path.open() as handle:
    for line in handle:
        if not line.strip():
            continue
        rows += 1
        tasks[json.loads(line).get('task', '<none>')] += 1

print('Dataset size GB:', round(dataset_path.stat().st_size / 1024**3, 3))
print('Rows:', rows)
print('Tasks:')
for task, count in tasks.most_common():
    print(f'  {task:<16} {count}')

# A mixture with no `task` keys means the old merged file was uploaded, which
# trains every row on the full field list instead of one task at a time.
assert '<none>' not in tasks, 'This file has no task tags — upload cleaned/task_mixture.jsonl'

## Resume Status

Read-only preview of exactly what the training cell will do: which checkpoint it picks,
which step it starts from, and whether the optimizer and LR schedule are restored.
Run it again any time — it never modifies anything.

In [ ]:
%%bash
export UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv
export UV_CACHE_DIR=/kaggle/temp/uv-cache
export NO_COLOR=1
uv run python - <<'PY'
import sys
sys.argv = ['resume-status', '--config', 'configs/train_qlora.yaml']
from qwen_cpp_review.cli import resume_status_main
resume_status_main()
PY

## Start or Resume Training

Detects the GPU count and launches single- or multi-GPU Accelerate. The trainer prints a
`RESUME MODE:` banner stating the starting step and whether the optimizer state, LR
schedule and step counter were restored — read that banner to confirm what happened.

If this session is killed, just re-run this cell (with `FRESH_START = False`).

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
export HF_HOME=/kaggle/temp/hf-cache
export HF_DATASETS_CACHE=/kaggle/temp/hf-datasets
export UV_CACHE_DIR=/kaggle/temp/uv-cache
export UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv
export ACCELERATE_LOG_LEVEL=info
export TRANSFORMERS_VERBOSITY=info
export NO_COLOR=1
LOG_FILE=/kaggle/working/train.log
echo "Training log: ${LOG_FILE}"
echo "Started at: $(date)" | tee -a "${LOG_FILE}"

NUM_GPUS=$(uv run python - <<'PY'
import torch
print(torch.cuda.device_count())
PY
)
echo "Detected GPUs: ${NUM_GPUS}" | tee -a "${LOG_FILE}"

if [ "${NUM_GPUS}" -gt 1 ]; then
  ACCEL_CONFIG=configs/accelerate_multi_gpu.yaml
  EXTRA_ARGS="--num_processes ${NUM_GPUS}"
else
  ACCEL_CONFIG=configs/accelerate_single_gpu.yaml
  EXTRA_ARGS=""
fi

uv run accelerate launch --config_file "${ACCEL_CONFIG}" ${EXTRA_ARGS} \
  train.py --config configs/train_qlora.yaml 2>&1 | tee -a "${LOG_FILE}"

## View Training Log

Run this after training finishes, or from the Kaggle console while training is running.

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv tail -n 80 /kaggle/working/train.log || true

## Check Saved Training Outputs

Expected outputs include `best_adapter/`, `best_adapter.pth`, `last_adapter/`, `last_adapter.pth`, `final_adapter/`, and `final_adapter.pth`.

In [ ]:
!ls -lah /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora || true
!find /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora -maxdepth 2 -type f \( -name '*.pth' -o -name 'adapter_model.safetensors' -o -name 'trainer_state.json' \) -print || true

## Evaluate Best Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python evaluate.py \
  --config configs/train_qlora.yaml \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter

## Smoke-Test the Trained Adapter

Runs easy/medium/hard C++ examples plus renamed-variable checks through the adapter that
`scripts/test_model.py` points at. Update the checkpoint path in that script to match the
adapter you want to test (`best_adapter`, `last_adapter`, or a specific `checkpoint-N`).

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python scripts/test_model.py
!head -2 /kaggle/working/outputs/model_test_predictions.jsonl || true

## Merge Best LoRA Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python merge_lora.py \
  --base-model Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter \
  --output-dir /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged

## Export Merged Model to ONNX

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python export_onnx.py \
  --model /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged \
  --output /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review.onnx

## Final Files to Download

Download from `/kaggle/working/outputs` after training finishes.

In [ ]:
!find /kaggle/working/outputs -maxdepth 3 -type f \( -name '*.pth' -o -name '*.onnx' -o -name 'adapter_model.safetensors' -o -name 'model.safetensors' -o -name 'training_config.yaml' \) -print || true